In [ ]:
# correct... this points used in earthlabel validation
import os 
import numpy as np
import geopandas as gpd
import pandas as pd
import rasterio
from shapely.geometry import Point, box
import folium

# ==========================================
# CONFIGURATION - ONLY IMD MAP IS REQUIRED
# ==========================================
FOLDER_PATH = r"C:\Users\user\OneDrive - Politecnico di Milano\File di Daniele Oxoli - test_embeddings\Matej_IMD_data"

IMD_PATH = os.path.join(FOLDER_PATH, "IMD_Milan_2018_reclass.tif")

# Target Outputs
OUTPUT_CSV = os.path.join(FOLDER_PATH, "test_outputs", "Milan_IMD_2018_points_test.csv")
OUTPUT_HTML = os.path.join(FOLDER_PATH, "test_outputs", "Milan_IMD_2018_points_map_test.html")

# Balanced Sampling Constraints
TILE_SIZE = 250            # Dimensions of systematic fishnet grids (meters)
SAMPLES_PER_STRATUM = 75   # Target balance quota per target class
CROSS_CLASS_MIN_DIST = 500  # Minimum distance enforced between DIFFERENT classes

COLOR_MAPPING = {
    "IMD_Class_0": "#006837",
    "IMD_Class_1": "#66bd63",
    "IMD_Class_2": "#d9ef8b",
    "IMD_Class_3": "#fee08b",
    "IMD_Class_4": "#f46d43",
    "IMD_Class_5": "#a50026"
}


def main():
    if not os.path.exists(IMD_PATH):
        print(f"[ERROR] Could not locate the required IMD raster layer at: {IMD_PATH}")
        return

    print("Opening IMD reference raster layer and extracting spatial metadata...")
    with rasterio.open(IMD_PATH) as imd_src:
        img = imd_src.read(1)
        transform = imd_src.transform
        nodata = imd_src.nodata
        bounds = imd_src.bounds
        crs = imd_src.crs
        
        # Calculate pixel dimensions
        pixel_size_x = abs(transform[0])
        pixel_size_y = abs(transform[4])
        
        # Create valid mask
        if nodata is not None and np.isnan(nodata):
            valid_mask = ~np.isnan(img)
        else:
            valid_mask = (img != nodata) & (~np.isnan(img))
            
        valid_pixel_count = np.sum(valid_mask)
        total_valid_area_m2 = valid_pixel_count * (pixel_size_x * pixel_size_y)

    # Calculate Expected Average Minimum Distance based on Area
    density = SAMPLES_PER_STRATUM / total_valid_area_m2
    calculated_avg_nn_dist = 1 / (2 * np.sqrt(density))
    SAME_CLASS_MIN_DIST = calculated_avg_nn_dist
    
    print("-" * 60)
    print(f"-> Total Valid Study Area: {total_valid_area_m2 / 1e6:.2f} sq km")
    print(f"-> Calculated Theoretical Class Avg NN Distance: {calculated_avg_nn_dist:.2f} meters")
    print("-" * 60)

    print("Extracting valid pixel coordinates using fast vectorized array lookup...")
    # Get row, col indices of all valid data points
    rows, cols = np.where(valid_mask)
    vals = img[rows, cols]
    
    # Standardize classes to integers
    vals_rounded = np.round(vals).astype(int)
    unique_classes = np.unique(vals_rounded)
    
    # Map row/col directly to spatial UTM coordinates using the affine transform
    # transform * (col, row) gives the top-left corner; add half pixel size for centroid
    x_coords, y_coords = rasterio.transform.xy(transform, rows, cols, offset='center')
    
    # Build a fast pandas DataFrame instead of a heavy GeoDataFrame first
    df_pixels = pd.DataFrame({
        'stratum_class': vals_rounded,
        'x': x_coords,
        'y': y_coords
    })
    
    print(f"Subdividing study window using a vector-free grid-binned index structure...")
    # Use mathematical grouping (floor division) to assign each point to a TILE_SIZE grid cell ID
    df_pixels['grid_x'] = ((df_pixels['x'] - bounds.left) // TILE_SIZE).astype(int)
    df_pixels['grid_y'] = ((df_pixels['y'] - bounds.bottom) // TILE_SIZE).astype(int)
    
    # Select 1 random point per class per grid tile using a fast pandas groupby operation
    print("Building raw grid-stratified sample candidate pool...")
    raw_sampled_df = (
        df_pixels.groupby(['grid_x', 'grid_y', 'stratum_class'])
        .sample(n=1, random_state=42)
        .reset_index(drop=True)
    )
    
    # Convert ONLY the final candidates into Geometry Point objects (very lightweight)
    geometries = [Point(x, y) for x, y in zip(raw_sampled_df['x'], raw_sampled_df['y'])]
    raw_tile_gdf = gpd.GeoDataFrame(raw_sampled_df, geometry=geometries, crs=crs)
    raw_tile_gdf['ref_label'] = "IMD_Class_" + raw_tile_gdf['stratum_class'].astype(str)

    # -------------------------------------------------------------------------
    # Spatial Pruning with Asymmetric Distance Constraints
    # -------------------------------------------------------------------------
    print(f"\nDownsampling to exactly {SAMPLES_PER_STRATUM} points per class...")
    final_accepted_gdf = gpd.GeoDataFrame(columns=["ref_label", "geometry", "stratum_class"], crs=crs)

    class_counts = raw_tile_gdf["stratum_class"].value_counts()
    sorted_classes = class_counts.sort_values().index.tolist()

    for target_class in sorted_classes:
        class_pool = raw_tile_gdf[raw_tile_gdf["stratum_class"] == target_class].copy()
        class_pool = class_pool.sample(frac=1, random_state=42)
        
        selected_points = []
        current_same_class_threshold = SAME_CLASS_MIN_DIST
        
        while len(selected_points) < SAMPLES_PER_STRATUM and current_same_class_threshold >= 100:
            selected_points = []
            for _, row in class_pool.iterrows():
                if len(selected_points) >= SAMPLES_PER_STRATUM:
                    break
                pt = row["geometry"]
                
                if not final_accepted_gdf.empty:
                    same_class_global = final_accepted_gdf[final_accepted_gdf["stratum_class"] == target_class]
                    diff_class_global = final_accepted_gdf[final_accepted_gdf["stratum_class"] != target_class]
                    
                    if not same_class_global.empty:
                        if (same_class_global.distance(pt) < current_same_class_threshold).any():
                            continue
                            
                    if not diff_class_global.empty:
                        if (diff_class_global.distance(pt) < CROSS_CLASS_MIN_DIST).any():
                            continue
                
                if selected_points:
                    local_distances = gpd.GeoSeries(selected_points, crs=crs).distance(pt)
                    if (local_distances < current_same_class_threshold).any():
                        continue
                        
                selected_points.append(pt)
                
            if len(selected_points) < SAMPLES_PER_STRATUM:
                current_same_class_threshold -= 10  

        # Fallback loop up to strict 100m floor
        if len(selected_points) < SAMPLES_PER_STRATUM:
            for _, row in class_pool.iterrows():
                if len(selected_points) >= SAMPLES_PER_STRATUM:
                    break
                pt = row["geometry"]
                
                if pt in selected_points:
                    continue
                    
                if not final_accepted_gdf.empty:
                    same_class_global = final_accepted_gdf[final_accepted_gdf["stratum_class"] == target_class]
                    diff_class_global = final_accepted_gdf[final_accepted_gdf["stratum_class"] != target_class]
                    
                    if not same_class_global.empty:
                        if (same_class_global.distance(pt) < 100).any():
                            continue
                    if not diff_class_global.empty:
                        if (diff_class_global.distance(pt) < CROSS_CLASS_MIN_DIST).any():
                            continue
                if selected_points:
                    if (gpd.GeoSeries(selected_points, crs=crs).distance(pt) < 100).any():
                        continue
                        
                selected_points.append(pt)

        class_rows = pd.DataFrame({
            "ref_label": f"IMD_Class_{target_class}",
            "geometry": selected_points,
            "stratum_class": target_class
        })
        final_accepted_gdf = pd.concat([final_accepted_gdf, gpd.GeoDataFrame(class_rows, crs=crs)], ignore_index=True)
        print(f"   -> Locked exactly {len(selected_points)} points for Class {target_class} (Final Same-Class Minimum: {max(current_same_class_threshold, 100):.1f}m | Diff-Class: {CROSS_CLASS_MIN_DIST}m)")

    # -------------------------------------------------------------------------
    # Format & Export Matrix (WGS84)
    # -------------------------------------------------------------------------
    print("\nRe-projecting final grid-stratified points to global WGS84 format (EPSG:4326)...")
    wgs84_gdf = final_accepted_gdf.to_crs(epsg=4326)

    df_out = pd.DataFrame()
    df_out["PLOTID"] = [f"PLOT_{i+1:03d}" for i in range(len(wgs84_gdf))]
    df_out["LON"] = wgs84_gdf["geometry"].x
    df_out["LAT"] = wgs84_gdf["geometry"].y
    df_out["ref_label"] = wgs84_gdf["ref_label"]

    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    df_out.to_csv(OUTPUT_CSV, index=False)
    print(f" Success! Balanced, cluster-free points saved to: {OUTPUT_CSV}")

    # -------------------------------------------------------------------------
    # Leaflet Visualization Map Generation
    # -------------------------------------------------------------------------
    print("Generating standalone interactive validation tracking map...")
    map_center = [df_out["LAT"].mean(), df_out["LON"].mean()]
    m = folium.Map(location=map_center, zoom_start=12, tiles="OpenStreetMap")

    for idx, row in df_out.iterrows():
        label = row["ref_label"]
        marker_color = COLOR_MAPPING.get(label, "#3186cc")
        
        folium.CircleMarker(
            location=[row["LAT"], row["LON"]],
            radius=5,
            popup=f"ID: {row['PLOTID']}<br>Strata: {label}",
            color=marker_color,
            fill=True,
            fill_color=marker_color,
            fill_opacity=0.85,
        ).add_to(m)

    unique_labels = sorted(df_out["ref_label"].unique())
    legend_html = """
    <div style="position: fixed; 
         top: 20px; right: 20px; width: 180px; height: auto; 
         border:2px solid grey; z-index:9999; font-size:14px;
         background-color:white; opacity: 0.9; padding: 10px;
         font-family: Arial, sans-serif; border-radius: 5px;">
         <b>IMD Class Legend</b><br>
    """
    for label in unique_labels:
        if label in COLOR_MAPPING:
            legend_html += f"""
             <div style="margin-top: 5px;">
             <i style="background:{COLOR_MAPPING[label]}; width: 14px; height: 14px; float: left; margin-right: 8px; border-radius: 50%;"></i>
             {label.replace('IMD_Class_', 'Class ')}<br>
             </div>
            """
    legend_html += "</div>"
    m.get_root().html.add_child(folium.Element(legend_html))

    m.save(OUTPUT_HTML)
    print(f" Success! Interactive map saved cleanly to: {OUTPUT_HTML}")


if __name__ == "__main__":
    main()